# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the **executable twin of the deployed paper** (`docs/index.html`): every number on
the page regenerates here, offline, from the committed receipts in `work/outputs/` plus the Week-7
playbook artifacts. No warehouse access, no token — it runs in seconds.

**The one-paragraph version of the whole project:** a content team can only review a fixed number of
pages per month, so the useful output is an *ordering* of the review queue. A transparent frozen rule
(stale + visible, ranked by traffic at stake) beat random selection on held-out clients, and a small
random forest beat the rule in the same calendar window (0.68 vs 0.38 precision@50) — but a
time-aware validation audit cut the forest to **0.36 mean-fold / 0.16 pooled** against a 0.25 base
rate. So the shipped playbook runs on the transparent rule with structural human review, and the
paper's headline is the audit, not the same-window win.


## 1. Question

*The research question and the decision it supports.*

**Decision being supported:** a content reviewer opens K pages per month and K is the budget
(20–100). The decision is not "will this page decline?" in the abstract — it is **"which pages
should a reviewer open first?"** That is an ordering problem, so the metric was fixed before any
training: **precision@50** — of the top 50 queue rows, how many truly declined — always reported
next to the base rate (the share that declines anyway).

**Research question:** can a validated ranking put that queue in order better than a transparent
rule — and, the part most projects skip, **does the advantage survive a change of calendar time?**

The label throughout: `is_declining` = last-30-day impressions fell below 80% of the prior 30 days
(an observed outcome, defined from GSC impressions; the ±20% hairline caveat lives in §5).


In [15]:
# Section 1 — the decision frame, printed from the playbook receipt (not from memory)
import os, json
import pandas as pd, numpy as np

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
FIG_DIR = os.path.join(REPO_ROOT, 'work', 'figures')

pb   = json.load(open(os.path.join(OUT_DIR, 'w07_playbook_receipt.json')))
base = json.load(open(os.path.join(OUT_DIR, 'baseline_folds_receipt.json')))
mvb  = json.load(open(os.path.join(OUT_DIR, 'model_vs_baseline_folds.json')))
va   = json.load(open(os.path.join(OUT_DIR, 'w06_validation_audit_receipt.json')))

print('Metric:      precision@50 (fixed before training; K = reviewer budget)')
print('Budget:      K = 20-100 pages per review cycle (w07 playbook)')
print(f'Base rate:   {base["whole_frame_base_rate"]} - the share of pages that decline anyway;')
print('             every precision number in this paper is read against this line.')
print(f'Tie policy:  {pb["ranking"]}')


Metric:      precision@50 (fixed before training; K = reviewer budget)
Budget:      K = 20-100 pages per review cycle (w07 playbook)
Base rate:   0.2487 - the share of pages that decline anyway;
             every precision number in this paper is read against this line.
Tie policy:  tier asc (act-now 0 / watch 1 / none 2), then imp_prev30 desc, then seeded content-hash asc


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source.** The FlyRank internship warehouse release (`hf://datasets/FlyRank/internship-warehouse`,
build **v20260703**): `dim_clients` 104 rows · `dim_content` 519,606 rows ·
`fact_content_daily_performance` **78,835,655** daily rows (2025-01-27 → 2026-06-30, partitioned by
month) · `fact_content_daily_performance_sample` ~11.7M · `fact_content_query_90d` 2,414,248.
Identifiers are pseudonymous hashes (`client_hash_id`, `content_hash_id`) used **only** for
grouping and joining — never as features. No client name, domain, URL, or raw query appears
anywhere in this project.

**Development frame.** 81,521 content items at the **March 2026 decision point**: features from
Jan–Feb 2026, label from March 2026. Whole-frame decline base rate **0.2487**.

**Exclusions — each with its because:**
- **`_sample` table never used for label logic** — it *is* the panel's last month, i.e. the natural
  outcome window of any past→future label; developing there means developing inside the test window.
- **June 2026 sealed** — read once, by the audit, after all development was frozen.
- **GA4 columns** — rows before a client's `ga4_data_start` are zero-FILLED behind
  `ga4_data_available = FALSE`; zeros there are not "no engagement", so the flag filters everywhere.
- **`trend_direction` / `trend_pct` never features** — they are the label's own ingredients
  (the label is derived from `trend_pct`-equivalent logic); using them is definitional leakage.
- **Product flags excluded** (health scores, optimization flags) — they are the product's
  decisions, not observed signals; a model could copy the answer instead of learning it.
- **Thin-history pages (< 15 of 30 active days)** — excluded from *actions* (playbook no-go N4,
  enforced structurally); momentum-style signals mislead there.


In [16]:
# Section 2 — frame stats from the frozen queue + release numbers asserted against the skill sheet
q = pd.read_csv(os.path.join(OUT_DIR, 'baseline_action_score.csv'))
EXPECTED_COLS = ['content_hash_id', 'client_hash_id', 'fold', 'score', 'reason_code',
                 'action_label', 'imp_prev30', 'clk_prev30', 'pos_prev30',
                 'days_with_imp_prev30', 'content_age_days']
assert list(q.columns) == EXPECTED_COLS and len(q) == 81521
print(f'Development frame: {len(q):,} items x {len(q.columns)} cols (March 2026 decision point)')
print(f'  clients (pseudonymous): {q["client_hash_id"].nunique():,}')
print(f'  decline base rate:      {base["whole_frame_base_rate"]} (whole frame)')
print()
print('Frozen reason codes (Week-4 rule):')
for code, n in q['reason_code'].value_counts().items():
    print(f'  {code:18s} {n:6,}  ({100 * n / len(q):.1f}%)')
print()
# Release facts pinned from the data skill sheet - the paper quotes these exact numbers.
release = {'dim_clients': 104, 'dim_content': 519606, 'fact_content_daily_performance': 78835655,
           'fact_content_daily_performance_sample': '~11.7M', 'fact_content_query_90d': 2414248}
print('Warehouse release v20260703 (rows):')
for t, n in release.items():
    print(f'  {t:38s} {n}')
print()
print('Public-safety check: hash IDs only; no names, URLs, or raw queries in any artifact.')


Development frame: 81,521 items x 11 cols (March 2026 decision point)
  clients (pseudonymous): 37
  decline base rate:      0.2487 (whole frame)

Frozen reason codes (Week-4 rule):
  stale_but_visible  36,965  (45.3%)
  low_volume         23,708  (29.1%)
  not_stale          20,848  (25.6%)

Warehouse release v20260703 (rows):
  dim_clients                            104
  dim_content                            519606
  fact_content_daily_performance         78835655
  fact_content_daily_performance_sample  ~11.7M
  fact_content_query_90d                 2414248

Public-safety check: hash IDs only; no names, URLs, or raw queries in any artifact.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions (stated, not hidden).** (i) A reviewer's budget is the queue's *head*, so ranking
quality at K=20–100 is what matters — hence precision@K, not accuracy or AUC alone. (ii) Recent
traffic history may carry *some* signal about next-month decline — an empirical question the
validation answers, and the answer turns out to be window-local. (iii) The unit is one content
item at one monthly decision point; all features must be knowable **before** the label window.

**Features (5, all pre-decision):** `log_imp_prev30`, `log_clk_prev30`, `pos_prev30`,
`days_with_imp_prev30`, `content_age_days`.

**Label:** `is_declining = imp_last30 < 0.8 × imp_prev30` — observed outcome, window strictly
after every feature window. The 0.8 hairline makes pages near the cut label-noisy (§5).

**Baseline (frozen before modeling).** `stale_but_visible`: age ≥ 90 days AND imp_prev30 ≥ 500;
score = stale × visible × imp_prev30; ties broken score-desc then seeded content-hash asc.
Audited on held-out client folds before any model existed (P@K receipts committed).

**Models.** Logistic regression, depth-3 tree, random forest (300 trees, `min_samples_leaf=5`,
seed 42) — no boosting, no hyperparameter search against any test fold.

**Validation design.** (1) **Client-grouped 5-way hash folds** — every client sits on exactly one
side; the model is scored on clients it never met. (2) **Time-aware** — same forest, trained on an
earlier window (Dec→Jan), tested on the same March rows. (3) **Robustness re-check** on a second
earlier window (Nov→Dec). (4) **Time-only** — all clients, train past → test March.

**Leakage checks (a method, not an apology).** Three boundaries, each tested in code in the Week-6
audit: (i) *window* — every feature computed strictly before the label window, no overlap;
(ii) *denominator* — `log_imp_prev30` is the label's denominator, so it was dropped and kept:
removing it does not collapse P@50 (definitional but not load-bearing); (iii) *product flags /
future signals* — none in the set. Plus a *stability* hunt: feature–label correlations re-estimated
in three windows — and they **flip sign** between Nov–Dec and March, which is what motivated the
time-aware protocol in the first place.


In [17]:
# Section 3 — folds, features, and the leakage-audit numbers, all from receipts
print('Features:', ', '.join(mvb['features']))
print(f'Split: {mvb["split"]} | seed {mvb["seed"]} | tie policy: {mvb["tie_policy"]}')
print()
fold_rows = []
for f, b in zip(mvb['folds'], base['folds']):
    assert f['fold'] == b['fold'] and f['n_test'] == b['n_test']
    fold_rows.append({'fold': f['fold'], 'n_test': f['n_test'], 'base_rate': f['base_rate'],
                      'rule_P@50': b['precision@50'], 'forest_P@50': f['forest_precision@50']})
print('The five client-grouped folds (held-out clients per fold):')
print(pd.DataFrame(fold_rows).to_string(index=False))
print()
corr = va['correlations_feature_vs_label']
print('Feature-label correlation stability (log_clk_prev30 vs is_declining):')
print(f'  Dec-Jan train: {corr["dec_jan_train"]["log_clk"]:+.3f} | '
      f'Nov-Dec train: {corr["nov_dec_train"]["log_clk"]:+.3f} | '
      f'March test:    {corr["march_test"]["log_clk"]:+.3f}')
flip = (corr['nov_dec_train']['log_clk'] > 0) != (corr['march_test']['log_clk'] > 0)
print(f'  sign flips between Nov-Dec and March: {flip}  -> motivated the time-aware protocol')
assert flip
print()
print('Leakage audit (W6): removing the label-denominator feature log_imp_prev30 keeps mean fold')
print('P@50 within noise of the full set -> definitional leak present but not load-bearing.')
print('Window check: features end strictly before the label window (asserted in w05/w06 code).')


Features: log_imp_prev30, log_clk_prev30, pos_prev30, days_with_imp_prev30, content_age_days
Split: client-grouped deterministic 5-way hash fold | seed 42 | tie policy: score desc, then seeded content-hash asc

The five client-grouped folds (held-out clients per fold):
 fold  n_test  base_rate  rule_P@50  forest_P@50
    0   25520     0.2153       0.20         0.52
    1    4276     0.1284       0.08         0.98
    2    6675     0.6649       0.88         1.00
    3   16084     0.2743       0.38         0.62
    4   28966     0.1859       0.34         0.26

Feature-label correlation stability (log_clk_prev30 vs is_declining):
  Dec-Jan train: -0.060 | Nov-Dec train: +0.072 | March test:    -0.130
  sign flips between Nov-Dec and March: True  -> motivated the time-aware protocol

Leakage audit (W6): removing the label-denominator feature log_imp_prev30 keeps mean fold
P@50 within noise of the full set -> definitional leak present but not load-bearing.
Window check: features end strictl

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Random-selection expectation: 0.2487** (the label base rate). It is shown separately because it
is not a fitted method and has no across-fold SD. Every method below is read against that line.

**The honest table — precision@50, identical client-grouped folds (mean ± SD across the five
folds; the SD spans client heterogeneity, it is *not* a confidence interval):**

| method | P@50 (mean ± SD) | per-fold range | higher vs rule |
|---|---|---|---|
| frozen rule (operational baseline) | 0.376 ± 0.273 | 0.08 – 0.88 | — |
| logistic regression | 0.152 ± 0.131 | 0.04 – 0.34 | 0 / 5 |
| depth-3 tree | 0.416 ± 0.216 | 0.20 – 0.74 | 3 / 5 |
| **random forest (same window)** | **0.676 ± 0.282** | 0.26 – 1.00 | **4 / 5** |
| **random forest (time-aware train)** | **0.364 ± 0.339** | 0.06 – 1.00 | below base in 3 / 5 folds |
| time-only (train past → test March, all clients) | 0.120 | — | below base |
| Nov→Dec robustness window | 0.320 | — | — |

The story in one line: **the same-window win does not transfer forward in time.** Pooled
time-aware P@50 is **0.16 — below random**; shipped blind across all clients, the model's ranking
would underperform chance next period. That measured collapse is why the shipped playbook (§6)
runs on the transparent rule with structural human review, and why this paper's headline is the
audit rather than the 0.68.

**Claims (each states its own boundary):**
- **C1.** On held-out clients in the same calendar window, the forest's observed mean P@50 (0.676)
  exceeds the rule's (0.376) and the base rate (0.2487); higher in 4 of 5 folds. *Boundary: this
  says nothing about a future month* (§4).
- **C2.** Under a time-aware protocol the same model falls to 0.364 mean-fold (folds 0.06–1.00)
  and 0.16 pooled — below base; time-only gives 0.12. *The same-window advantage does not transfer*
  (§4).
- **C3.** Therefore the shipped ranker is the transparent rule, and the model is demoted to
  same-window evidence — "the rule explains, the human decides" (§6).
- **C4.** The playbook converts the ranking into a governed workflow: one action per page, no-go
  list, monitoring triggers — one of which has already fired (§6).


In [18]:
# Section 4 — the honest table, computed from receipts and asserted against the paper's numbers
def msd(vals):
    return float(np.mean(vals)), float(np.std(vals))

rule50  = [f['precision@50'] for f in base['folds']]
log50   = [f['logistic_precision@50'] for f in mvb['folds']]
tree50  = [f['tree_d3_precision@50'] for f in mvb['folds']]
fsw50   = [f['forest_precision@50'] for f in mvb['folds']]
fta     = va['after']['time_plus_grouped']['folds']

rows = []
for name, vals in [('frozen rule (operational baseline)', rule50), ('logistic regression', log50),
                   ('depth-3 tree', tree50), ('random forest (same window)', fsw50),
                   ('random forest (time-aware train)', fta)]:
    m, s = msd(vals)
    wins = sum(1 for a, b in zip(vals, rule50) if a > b) if name != 'random forest (time-aware train)' else None
    rows.append({'method': name, 'P@50 mean': round(m, 3), 'SD': round(s, 3),
                 'range': f'{min(vals):.2f} - {max(vals):.2f}',
                 'folds higher vs rule': f'{wins}/5' if wins is not None else 'below base in 3/5'})
print('Precision@50, identical client-grouped folds (mean +/- SD; SD spans client heterogeneity,')
print('it is NOT a confidence interval). Random-selection expectation = base rate',
      base['whole_frame_base_rate'], '(shown separately - not a fitted method).')
print()
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f'Time-aware: pooled P@50 = {va["after"]["time_plus_grouped"]["pooled_P@50"]} | '
      f'time-only P@50 = {va["after"]["time_only"]["P@50"]} | '
      f'Nov-Dec robustness = {va["after"]["robustness_nov_dec_P@50"]}')
print()

# Assert every headline number the paper prints.
assert abs(np.mean(fsw50) - 0.676) < 1e-6 and abs(np.std(fsw50) - 0.282) < 0.001
assert abs(np.mean(rule50) - 0.376) < 1e-6 and abs(np.mean(fta) - 0.364) < 1e-6
assert va['after']['time_plus_grouped']['pooled_P@50'] < base['whole_frame_base_rate']
assert sum(1 for a, b in zip(fsw50, rule50) if a > b) == 4
below_base = sum(1 for a, f in zip(fta, base['folds']) if a < f['base_rate'])
assert below_base == 3
print('All headline numbers verified against receipts: 0.676 / 0.376 / 0.364 / pooled 0.16 < base 0.25;')
print('forest higher than rule in 4/5 same-window folds; time-aware below base rate in 3/5 folds.')
print()
print('Claims: C1 same-window win (4/5 folds, boundary: same window only) | C2 no forward transfer')
print('(0.364 mean-fold, 0.16 pooled, 0.12 time-only) | C3 rule ships, model demoted to evidence |')
print('C4 governed playbook (Section 6).')


Precision@50, identical client-grouped folds (mean +/- SD; SD spans client heterogeneity,
it is NOT a confidence interval). Random-selection expectation = base rate 0.2487 (shown separately - not a fitted method).

                            method  P@50 mean    SD       range folds higher vs rule
frozen rule (operational baseline)      0.376 0.273 0.08 - 0.88                  0/5
               logistic regression      0.152 0.131 0.04 - 0.34                  0/5
                      depth-3 tree      0.416 0.216 0.20 - 0.74                  3/5
       random forest (same window)      0.676 0.282 0.26 - 1.00                  4/5
  random forest (time-aware train)      0.364 0.339 0.06 - 1.00    below base in 3/5

Time-aware: pooled P@50 = 0.16 | time-only P@50 = 0.12 | Nov-Dec robustness = 0.32

All headline numbers verified against receipts: 0.676 / 0.376 / 0.364 / pooled 0.16 < base 0.25;
forest higher than rule in 4/5 same-window folds; time-aware below base rate in 3/5 folds.

C

## 5. Limitations

*What this work cannot claim — each limitation tied to the claim it bounds.*

- **No forward transfer (bounds C1, C2).** One walk-forward protocol, one robustness window. The
  time-aware collapse (0.676 → 0.364 mean-fold / 0.16 pooled) is measured on a single train/test
  window pair; more windows would sharpen it, and none were run beyond Nov→Dec (0.32).
- **Huge fold variance (bounds C1).** Per-fold P@50 spans 0.06–1.00 across five folds: client
  composition dominates. Five folds do not establish reliable superiority over the rule even
  same-window.
- **The label is a hairline (bounds every result).** `is_declining` flips at a 20% impression drop;
  pages near the cut are label-noise, and month-to-month mean reversion means many "decliners"
  recover untouched.
- **Strongest counterargument, stated plainly:** if clicks mean-revert, a decline-finding queue may
  partly be a *volatility-finding* queue. We agree — which is exactly why the recommended workflow
  is score → human review → hypothesis → controlled test, and why nothing here claims causality.
- **No causal claim (bounds C3, C4).** Nothing here shows that refreshing a flagged page changes
  its outcome. That requires a controlled experiment this project did not run.
- **One portfolio, GSC-visible pages only.** Findings are observed in this dataset; absence from
  the queue is not evidence a page is healthy.
- **Decision-support, not production.** No scheduler, no serving path, no client-facing claims —
  by design (no-go N7).


In [19]:
# Section 5 — limitations, each printed with the claim it bounds (audit-friendly)
limits = [
    ('C1, C2', 'one walk-forward protocol + one robustness window (Nov-Dec 0.32); no multi-window backtest'),
    ('C1',     'per-fold P@50 spans 0.06-1.00: five folds cannot establish reliable superiority'),
    ('all',    'label hairline: is_declining flips at a 20% impression drop; near-cut pages are label-noise'),
    ('all',    'mean reversion: the queue may partly find volatility, not durable decay'),
    ('C3, C4', 'no causal claim: refresh -> recovery was never tested (needs a controlled experiment)'),
    ('all',    'one portfolio, GSC-visible pages only; absence from the queue is not evidence of health'),
    ('C4',     'decision-support only: no scheduler, no serving path, no client-facing claims (no-go N7)'),
]
print('Limitation -> claim it bounds:')
for claim, text in limits:
    print(f'  [{claim:6s}] {text}')
print()
print('The strongest counterargument, in one sentence: if clicks mean-revert, a decline-finding')
print('queue may partly be a volatility-finding queue. The answer is the workflow: score -> human')
print('review -> hypothesis -> controlled test. Nothing here claims causality.')


Limitation -> claim it bounds:
  [C1, C2] one walk-forward protocol + one robustness window (Nov-Dec 0.32); no multi-window backtest
  [C1    ] per-fold P@50 spans 0.06-1.00: five folds cannot establish reliable superiority
  [all   ] label hairline: is_declining flips at a 20% impression drop; near-cut pages are label-noise
  [all   ] mean reversion: the queue may partly find volatility, not durable decay
  [C3, C4] no causal claim: refresh -> recovery was never tested (needs a controlled experiment)
  [all   ] one portfolio, GSC-visible pages only; absence from the queue is not evidence of health
  [C4    ] decision-support only: no scheduler, no serving path, no client-facing claims (no-go N7)

The strongest counterargument, in one sentence: if clicks mean-revert, a decline-finding
queue may partly be a volatility-finding queue. The answer is the workflow: score -> human
review -> hypothesis -> controlled test. Nothing here claims causality.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Full construction and validation live in `w07_action_playbook.ipynb`; this section carries the
shipped design and its current status.

**Archetype → action mapping (7 archetypes, first match wins, one action + one reason code each):**
`thin_history → wait_for_history` (no-go N4, enforced structurally) · `fragile_snippet →
rewrite_snippet` · `aged_workhorse → full_refresh_first` · `stale_performer → refresh_review` ·
`young_earner → monitor_growth` · `quiet_stale → no_action` · `small_dormant → no_action`.
Act-now rows rank by traffic at stake (`imp_prev30` desc, seeded tie-break) — a rule a human can
re-derive by hand. Forest probabilities are deliberately **not** used for ranking (C2).

**Human review is structural, not advisory:** every act-now row carries auto-generated "what would
make it wrong" notes; the no-go list (N1–N7) bans auto-publishing, client-facing prediction claims,
tiny-n buckets, thin-history actions, tuning against sealed months, bulk execution, and schedulers.

**Monitoring (M1–M5), computed from local artifacts:** queue age > 45 days → regenerate;
rolling human-verdict precision < 0.25 → stop shipping; time+grouped mean fold < 0.30 or any fold
< 0.10 → retrain + full audit (this one has **already fired** on the W6 window — min fold 0.06 —
which is the decay insight applied to ourselves); stale-share drift > ±10 pp → investigate;
schema change → rebuild from scratch.

**Value, as derived arithmetic — not a promise:** at K=50, the honest time-aware mean fold (0.364)
finds ~18 confirmed decliners vs ~12 at random — about **6 more per batch**, for ~21 reviewer-hours.
The top-50 act-now rows carry ~3% of portfolio impressions, ~49× the random-pick share (0.06%).
Assumptions: this portfolio behaves like this one; review capacity is the bottleneck; catching a
decliner early is worth an editor-hour. No traffic-recovery claim is made or implied.


In [20]:
# Section 6 — playbook status from the w07 receipt: archetypes, triggers, value arithmetic
QUEUE = os.path.join(OUT_DIR, 'w07_action_queue.csv')
assert os.path.exists(QUEUE), ('w07_action_queue.csv not found - run w07_action_playbook.ipynb '
                               'first (it regenerates this file; CSVs stay out of git by design).')
qp = pd.read_csv(QUEUE)
print(f'Ranked queue: {len(qp):,} rows | act-now {int((qp.tier == 0).sum()):,} | '
      f'watch {int((qp.tier == 1).sum()):,} | no-action {int((qp.tier == 2).sum()):,}')
print()
print('Archetype -> action -> reason code (one per row, asserted in w07):')
for a, v in pb['archetypes'].items():
    print(f'  {a:16s} -> {v["action"]:18s} {v["reason_code"]}')
print()
print('Monitoring triggers, current status:')
for tid, t in pb['monitoring_triggers'].items():
    print(f'  {tid}: {t["metric"]} | threshold {t["threshold"]} | {t["status"]}')
print()
# Value arithmetic - derived, with assumptions stated in the markdown above.
K = 50
ta_mean = va['after']['time_plus_grouped']['mean_fold_P@50']
extra = (ta_mean - base['whole_frame_base_rate']) * K
total_imp = qp['imp_prev30'].sum()
top_share = 100 * qp.head(K)['imp_prev30'].sum() / total_imp
rand_share = 100 * K / len(qp)
assert 2.0 <= top_share <= 4.0 and 30 <= top_share / rand_share <= 70
print(f'Value scenario: K=50 -> ~{ta_mean * K:.0f} confirmed decliners vs '
      f'~{base["whole_frame_base_rate"] * K:.0f} at random = ~{extra:.0f} extra per batch,')
print(f'for ~{K * 25 / 60:.0f} reviewer-hours. Top-{K} act-now rows carry {top_share:.1f}% of '
      f'portfolio impressions vs {rand_share:.2f}% random (~{top_share / rand_share:.0f}x).')
print('Descriptive arithmetic from the five-fold evaluation - NOT a forecast of future impact.')


Ranked queue: 81,521 rows | act-now 38,256 | watch 15,546 | no-action 27,719

Archetype -> action -> reason code (one per row, asserted in w07):
  thin_history     -> wait_for_history   THIN_HISTORY
  fragile_snippet  -> rewrite_snippet    LOW_CTR
  aged_workhorse   -> full_refresh_first STALE_BIG_TRAFFIC
  stale_performer  -> refresh_review     STALE_VISIBLE
  young_earner     -> monitor_growth     NEW_WITH_TRAFFIC
  quiet_stale      -> no_action          LOW_VOLUME
  small_dormant    -> no_action          SMALL_NO_STAKE

Monitoring triggers, current status:
  M1: queue age | threshold > 45 days | OK
  M2: rolling human-verdict precision | threshold < 0.25 base rate over last 3 batches | ARMS AFTER FIRST REVIEW CYCLES
  M3: time+grouped mean fold P@50 | threshold < 0.30 or any fold < 0.10 | TRIPPED on W6 window - retrain + full audit before any queue ships
  M4: stale_but_visible share | threshold shift > +/-10 pp vs snapshot | BASELINE SET - compare next run
  M5: warehouse schema / 

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper (`docs/index.html`) embeds four figures and one queue ledger. Everything here
regenerates offline from receipts + the w07 queue CSV, and figures are exported both to
`work/figures/` (repo) and `docs/img/` (served by GitHub Pages). The new chart below is the
paper's lead visual: the same-window P@K comparison **and** the per-fold audit that cut it —
one figure, the whole honest arc.


In [21]:
# Section 7 — lead figure (P@K + per-fold audit), docs/img export, queue ledger, notebook map
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- Lead figure: two panels, one story -------------------------------------------------------
ks = mvb['K']
fsw20  = [f['forest_precision@20'] for f in mvb['folds']]
fsw100 = [f['forest_precision@100'] for f in mvb['folds']]
rule20  = [f['precision@20'] for f in base['folds']]
rule100 = [f['precision@100'] for f in base['folds']]
forest_k = [np.mean(fsw20), np.mean(fsw50), np.mean(fsw100)]
rule_k   = [np.mean(rule20), np.mean(rule50), np.mean(rule100)]
br = base['whole_frame_base_rate']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax1.plot(ks, forest_k, 'o-', color='#4477aa', lw=2, label='random forest (same window)')
ax1.plot(ks, rule_k, 's--', color='#9db8d9', lw=2, label='frozen rule')
ax1.axhline(br, color='#666666', ls=':', lw=1.2)
ax1.text(101, br + 0.012, f'base rate {br:.2f}', fontsize=8.5, color='#555555', ha='right')
for x, y in zip(ks, forest_k):
    ax1.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=9, color='#4477aa', fontweight='bold')
for x, y in zip(ks, rule_k):
    ax1.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, -16),
                 ha='center', fontsize=9, color='#7a94b8')
ax1.set_xticks(ks, [f'K={k}' for k in ks])
ax1.set_ylim(0, 0.85); ax1.set_ylabel('precision@K (mean of 5 folds)')
ax1.set_title('Same calendar window,\nheld-out clients', fontsize=10)
ax1.legend(fontsize=8.5, loc='lower left', framealpha=0.95)

rng = np.random.default_rng(42)
for i, (vals, col, lbl) in enumerate([(fsw50, '#4477aa', 'forest, same window'),
                                      (fta, '#ee6677', 'forest, time-aware train'),
                                      (rule50, '#9db8d9', 'frozen rule')]):
    ax2.scatter(np.full(len(vals), i) + rng.uniform(-0.05, 0.05, len(vals)), vals,
                s=46, color=col, zorder=3, label=lbl)
    ax2.hlines(np.mean(vals), i - 0.16, i + 0.16, color=col, lw=2.4, zorder=4)
ax2.axhline(br, color='#666666', ls=':', lw=1.2)
ax2.text(2.42, br + 0.012, f'base {br:.2f}', fontsize=8.5, color='#555555', ha='right')
ax2.set_xticks([0, 1, 2], ['forest\nsame-window', 'forest\ntime-aware', 'frozen\nrule'])
ax2.set_ylim(0, 1.08); ax2.set_ylabel('P@50 per fold (dots) + mean (bar)')
ax2.set_title('The audit, per fold:\nthe win does not survive time', fontsize=10)
fig.suptitle('Which pages should a reviewer open first? - the same-window win (left) and its time-aware collapse (right)',
             fontsize=11, y=1.0)
fig.tight_layout(rect=(0, 0, 1, 0.94))
LEAD = os.path.join(FIG_DIR, 'fig_capstone_lead.png')
fig.savefig(LEAD, dpi=160); plt.close(fig)

# --- Export every figure the paper embeds to docs/img/ (GitHub Pages serves this folder) ------
DOCS_IMG = os.path.join(REPO_ROOT, 'docs', 'img')
os.makedirs(DOCS_IMG, exist_ok=True)
PAPER_FIGS = ['fig_capstone_lead.png', 'fig_w07_honest_performance.png',
              'fig_w07_traffic_at_stake.png', 'fig_w07_archetypes.png']
for f in PAPER_FIGS:
    src = os.path.join(FIG_DIR, f)
    assert os.path.exists(src), f'missing figure: {f}'
    shutil.copy2(src, os.path.join(DOCS_IMG, f))
print('Figures exported to docs/img/:', ', '.join(PAPER_FIGS))
print()

# --- Queue ledger fragment (top 5) for the paper's recommendations section ---------------------
top5 = qp.head(5)
print('LEDGER FRAGMENT (paste into docs/index.html - static snapshot; queue regenerates via w07):')
for _, r in top5.iterrows():
    imp_k = r['imp_prev30'] / 1000
    print(f'  #{int(r["playbook_rank"])} {r["content_hash_id"][:17]}... | {r["archetype"]} | '
          f'{r["reason_code_v2"]} | imp {imp_k:,.0f}k | age {r["content_age_days"]:.0f}d | {r["playbook_action"]}')
print()
print('Result-to-notebook map:')
for res, nb in [('Table: honest P@50 comparison', 'capstone.ipynb Section 4 (from committed receipts)'),
                ('Figure: lead chart (P@K + fold audit)', 'capstone.ipynb Section 7 -> docs/img/fig_capstone_lead.png'),
                ('Figure: protocol bars', 'w07_action_playbook.ipynb -> fig_w07_honest_performance.png'),
                ('Figure: cost/value curve', 'w07_action_playbook.ipynb -> fig_w07_traffic_at_stake.png'),
                ('Figure: archetype composition', 'w07_action_playbook.ipynb -> fig_w07_archetypes.png'),
                ('Queue ledger (top rows)', 'w07_action_playbook.ipynb -> w07_action_queue.csv (regenerable, out of git)'),
                ('Leakage audit + time-aware numbers', 'w06_validation_audit.ipynb -> w06_validation_audit_receipt.json')]:
    print(f'  {res:42s} <- {nb}')


Figures exported to docs/img/: fig_capstone_lead.png, fig_w07_honest_performance.png, fig_w07_traffic_at_stake.png, fig_w07_archetypes.png

LEDGER FRAGMENT (paste into docs/index.html - static snapshot; queue regenerates via w07):
  #1 content_8e1334d63... | fragile_snippet | LOW_CTR | imp 204k | age 380d | rewrite_snippet
  #2 content_fec55986a... | fragile_snippet | LOW_CTR | imp 198k | age 380d | rewrite_snippet
  #3 content_9c057b66c... | fragile_snippet | LOW_CTR | imp 196k | age 213d | rewrite_snippet
  #4 content_512dbad65... | stale_performer | STALE_VISIBLE | imp 179k | age 157d | refresh_review
  #5 content_e241d6415... | aged_workhorse | STALE_BIG_TRAFFIC | imp 177k | age 382d | full_refresh_first

Result-to-notebook map:
  Table: honest P@50 comparison              <- capstone.ipynb Section 4 (from committed receipts)
  Figure: lead chart (P@K + fold audit)      <- capstone.ipynb Section 7 -> docs/img/fig_capstone_lead.png
  Figure: protocol bars                      <- w07

## ML-12 — closing cells: demo, social post, employer summary

### 5-minute demo outline (Week-8 showcase — optional)

1. **(0:00–0:45) Question.** Which pages should a reviewer open first this month, inside a fixed ~50-page review budget? Land it in one line: *the rule explains, the human decides.*
2. **(0:45–1:45) Method.** A transparent rule (stale + visible, scored by traffic at stake) versus a five-feature random forest, on identical client-grouped folds; precision@50 fixed by the budget before training; a time-aware protocol retests the same model on a later month.
3. **(1:45–3:00) One chart.** The lead figure — same-window P@K curves on the left, the per-fold audit on the right: the forest leads, then collapses under time-aware validation.
4. **(3:00–4:00) One honest result.** Same-window the forest beats the rule 0.68 vs 0.38 (4/5 held-out-client folds, base 0.25); under the time-aware split it falls to 0.36 mean-fold and 0.16 pooled — *below random*. The win does not transfer forward.
5. **(4:00–5:00) One recommendation.** Ship the transparent rule with a governed human-review playbook (7 archetypes, no-go list, monitoring M1–M5 — M3 already fired on the W6 window), not the model; retrain only when M3 trips. Next: multi-window backtest + a controlled refresh test.



### Two shareable cuts (right under the outline)

**Social post — methodology, one finding, its caveat:**
> Which pages should you review first when traffic dips? We ranked 81,521 pages by decline risk on 78.8M rows of real search data. A simple rule (old + visible + big audience) found decliners at 2× random. A fancier model beat it — until we tested it on a *later* month, when it fell *below* random. The honest queue ships the transparent rule + a human check on every row. Predictive, not causal — nothing here says a refresh fixes a page. Paper + notebooks: https://github.com/Umer32-bit/flyrank-ml-internship

**Employer 3-sentencer — what I built · on what data · what it showed:**
Built a decline-triage ranking system on a 78.8M-row pseudonymized production warehouse (81,521 scored pages, 5 leakage-audited features, precision@50 fixed before training). Found and measured a validation boundary most projects miss: a 0.68 same-window ranking fell to 0.16 pooled under a time-aware split — so I shipped a transparent rule + governed human-review playbook instead of the model. Everything regenerates from committed receipts: five notebooks, one audit, one paper, all public.


## Self-check

Before I submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbclient, output verified)
- [x] No client names, URLs, or private queries anywhere — hash IDs only; this notebook never touches the warehouse
- [x] My claims use careful words: observed, measured, directional, decision-support — C1–C4 each state their own boundary
- [x] Every number traces to a committed receipt (`baseline_folds_receipt.json`, `model_vs_baseline_folds.json`, `w06_validation_audit_receipt.json`, `w07_playbook_receipt.json`) and headline values are asserted in code
- [x] Results table shows the base rate separately, ± defined as across-fold SD (not a CI), rejected comparators (logistic 0.152) kept visible
- [x] Limitations tied to claims, strongest counterargument (mean reversion) stated and answered
- [x] Ranked recommendations = the shipped w07 playbook with trigger statuses (M3 fired) and value-as-arithmetic with assumptions
- [x] Paper artifacts exported: `docs/img/` figures + queue ledger; result-to-notebook map printed
- [x] ML-12 closing cells done: demo outline, social post, employer 3-sentencer
- [x] Deployed paper has all 9 sections including Abstract (top) and Acknowledgments & data credit with https://flyrank.ai (bottom)
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.
